In [1]:
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import KFold, cross_validate, train_test_split
from xgboost import XGBRegressor

# ==============================================================================
# 1. HYPERPARAMETER
# ==============================================================================
xgb_params = {
    "n_estimators": 500,
    "max_depth": 4,
    "learning_rate": 0.05,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "objective": "reg:squarederror",
    "random_state": 42,
    "n_jobs": -1,
}

# ==============================================================================
# 2. DATENSÄTZE LADEN & BEREINIGEN
# ==============================================================================
datasets = {
    "1. Ohne Geo": pd.read_pickle("df_model_base_no_geo.pkl"),
    "2. OHE Nachbarschaften": pd.read_pickle("df_model_neighbourhood_ohe.pkl"),
    "3. Target Encoding (OOF)": pd.read_pickle("df_model_target_encoded.pkl"),
}

leakage_cols = [
    "review_scores_rating",
    "review_scores_location",
    "review_neighbourhood_rating",
    "neighbourhood_cleansed",
]

results = []

# ==============================================================================
# 3. EVALUATIONS-SCHLEIFE (HOLDOUT + 5-FOLD CV)
# ==============================================================================
print("Starte Modellvergleich (Holdout-Test & 5-Fold-CV)...\n")

for name, df in datasets.items():
    # Zielvariable und Prädiktorenmatrix
    y = df["review_scores_rating"]
    drop_cols = [c for c in leakage_cols if c in df.columns]
    X = df.drop(columns=drop_cols)

    # --- A. HOLDOUT-SPLIT (60 - 20 - 20) ---
    X_temp, X_test, y_temp, y_test = train_test_split(
        X, y, test_size=0.20, random_state=42
    )
    X_train, X_val, y_train, y_val = train_test_split(
        X_temp, y_temp, test_size=0.25, random_state=42
    )

    model = XGBRegressor(**xgb_params)
    model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)

    preds_test = model.predict(X_test)
    test_r2 = r2_score(y_test, preds_test)
    test_mae = mean_absolute_error(y_test, preds_test)

    # --- B. 5-FACHE CROSS-VALIDATION ---
    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    cv_out = cross_validate(
        XGBRegressor(**xgb_params),
        X,
        y,
        cv=kf,
        scoring={"r2": "r2", "mae": "neg_mean_absolute_error"},
        n_jobs=-1,
    )

    cv_r2_mean = cv_out["test_r2"].mean()
    cv_r2_std = cv_out["test_r2"].std()
    cv_mae_mean = -cv_out["test_mae"].mean()
    cv_mae_std = -cv_out["test_mae"].std()

    # Ergebnisse speichern
    results.append(
        {
            "Modell": name,
            "Features": X.shape[1],
            "Test R² (Split)": round(test_r2, 4),
            "Test MAE (Split)": round(test_mae, 4),
            "CV R² (Mean ± SD)": f"{cv_r2_mean:.4f} (± {cv_r2_std:.4f})",
            "CV MAE (Mean ± SD)": f"{cv_mae_mean:.4f} (± {cv_mae_std:.4f})",
        }
    )

# ==============================================================================
# 4. GESAMTTABELLE AUSGEBEN
# ==============================================================================
summary_df = pd.DataFrame(results).set_index("Modell")
print("=" * 85)
print("=== GESAMTVERGLEICH DER MODELLE (HOLDOUT vs. 5-FOLD CV) ===")
print("=" * 85)
print(summary_df.to_string())

Starte Modellvergleich (Holdout-Test & 5-Fold-CV)...

=== GESAMTVERGLEICH DER MODELLE (HOLDOUT vs. 5-FOLD CV) ===
                          Features  Test R² (Split)  Test MAE (Split)  CV R² (Mean ± SD)  CV MAE (Mean ± SD)
Modell                                                                                                      
1. Ohne Geo                     29           0.2956            0.1411  0.3418 (± 0.0335)  0.1373 (± -0.0033)
2. OHE Nachbarschaften         164           0.3260            0.1382  0.3542 (± 0.0351)  0.1360 (± -0.0034)
3. Target Encoding (OOF)        30           0.2860            0.1417  0.3357 (± 0.0323)  0.1377 (± -0.0037)
